# 03 Statistics, probability and A/B test

For every test: question, H0 and H1, the test and why, assumptions, alpha = 0.05, statistic, p-value, effect size,
and a plain-English conclusion. All numbers are printed by code.

| Part | Data |
|---|---|
| 1. Welch t-test: review score, late vs on-time | **real** (Olist) |
| 2. ANOVA / Kruskal-Wallis: order value by payment type | **real** |
| 3. Chi-square: RFM segment vs region | **real** |
| 4. Probability: buy again, low review given late | **real** |
| 5. A/B test: free vs paid shipping | **SIMULATED data** (made up with a random seed, not real customers) |

A small p-value only says that a difference is unlikely to be chance. With about 100,000 orders almost every difference
becomes "significant", so each test also reports an **effect size** to show if the difference is big enough to matter.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy import stats

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

pd.set_option("display.width", 200)
ALPHA = 0.05
PROC, OUT = ROOT / "data" / "processed", ROOT / "sql" / "outputs"


def say(text):
    display(Markdown(text))


def pfmt(p):
    return "< 1e-300" if p < 1e-300 else f"{p:.3g}"


def decide(p):
    return "reject H0" if p < ALPHA else "do not reject H0"


def q(prefix):
    return pd.read_csv(next(OUT.glob(f"{prefix}_*.csv")))


orders = pd.read_parquet(PROC / "fact_orders.parquet")
print("orders:", len(orders))

orders: 99441


## 1. Welch t-test: do on-time deliveries get higher review scores than late ones? (real data)

- **Question:** Is the average review score of on-time (or early) orders higher than that of late orders?
- **H0:** the two groups have the same mean review score.  **H1:** on-time orders have a higher mean (one-sided).
- **Test:** Welch t-test. It compares two means and does **not** assume equal variances (Levene's test below checks this).
  With thousands of orders per group, the t-test is fine even though scores are 1 to 5 (central limit theorem).
  A Mann-Whitney test is added as a check, because scores are ordinal.
- **Data:** delivered orders with a delivery date and a review (the same group as Q14). Late = delivered after the estimated date.

In [2]:
d = orders[(orders.order_status == "delivered") & orders.delay_days.notna() & orders.review_score.notna()].copy()
d["review_score"] = d.review_score.astype(float)
ontime = d.loc[d.delay_days <= 0, "review_score"]
late = d.loc[d.delay_days > 0, "review_score"]

# The group sizes and means must match the SQL result of Q14
q14 = q("q14")
q14 = q14[q14.level == "late_vs_on_time"].set_index("group_name")
assert len(late) == q14.loc["Late", "orders"] and len(ontime) == q14.loc["On time or early", "orders"]
assert abs(late.mean() - q14.loc["Late", "avg_review_score"]) < 1e-3
print("Group sizes and means match Q14.")

desc = pd.DataFrame({"n": [len(ontime), len(late)], "mean": [ontime.mean(), late.mean()],
                     "sd": [ontime.std(), late.std()], "median": [ontime.median(), late.median()]},
                    index=["On time or early", "Late"]).round(4)
display(desc)
print("Share of each score (%):")
display((pd.concat({"On time or early": ontime.value_counts(normalize=True),
                    "Late": late.value_counts(normalize=True)}, axis=1).sort_index() * 100).round(1))

Group sizes and means match Q14.


,n,mean,sd,median
On time or early,88169,4.2941,1.1472,5.0
Late,7655,2.5637,1.6582,2.0


Share of each score (%):


,On time or early,Late
review_score,,
1.0,6.6,46.2
2.0,2.6,7.9
3.0,8.0,11.4
4.0,20.4,12.3
5.0,62.4,22.2


In [3]:
lev = stats.levene(ontime, late, center="median")
print(f"Assumption check, Levene (equal variances): statistic = {lev.statistic:.1f}, p = {pfmt(lev.pvalue)}"
      f" -> variances differ, so Welch (not Student) is the right choice.")

res = stats.ttest_ind(ontime, late, equal_var=False, alternative="greater")      # H1: on time > late
res_two = stats.ttest_ind(ontime, late, equal_var=False)                          # two-sided, for the CI
ci = res_two.confidence_interval(0.95)
diff = ontime.mean() - late.mean()
sp = np.sqrt(((len(ontime) - 1) * ontime.var() + (len(late) - 1) * late.var()) / (len(ontime) + len(late) - 2))
cohen_d = diff / sp

mw = stats.mannwhitneyu(ontime, late, alternative="greater")
prob_superiority = mw.statistic / (len(ontime) * len(late))   # P(random on-time score > random late score), ties count 1/2

def d_label(d):
    d = abs(d)
    return "negligible" if d < 0.2 else "small" if d < 0.5 else "medium" if d < 0.8 else "large"

say(f'''
**Result (alpha = {ALPHA})**

| Item | Value |
|---|---|
| Difference in means (on time minus late) | {diff:.3f} score points |
| 95% confidence interval of the difference | {ci.low:.3f} to {ci.high:.3f} |
| Welch t statistic, degrees of freedom | {res.statistic:.1f}, {res.df:,.0f} |
| p-value (one-sided) | {pfmt(res.pvalue)} |
| Effect size, Cohen's d | {cohen_d:.2f} ({d_label(cohen_d)}; rule of thumb: 0.2 small, 0.5 medium, 0.8 large) |
| Mann-Whitney p-value (check) | {pfmt(mw.pvalue)} |
| Probability that a random on-time order has a higher score than a random late order (ties count half) | {prob_superiority:.3f} |

**Decision: {decide(res.pvalue)}.** On-time orders score {diff:.2f} points higher on average ({ontime.mean():.2f} vs {late.mean():.2f}).
The difference is statistically clear and the effect size is {d_label(cohen_d)} (d = {cohen_d:.2f}), so it also matters in practice.
The Mann-Whitney test gives the same answer. *Limit:* this shows a link, not proof that lateness alone causes the low scores.
''')

Assumption check, Levene (equal variances): statistic = 3373.5, p = < 1e-300 -> variances differ, so Welch (not Student) is the right choice.



**Result (alpha = 0.05)**

| Item | Value |
|---|---|
| Difference in means (on time minus late) | 1.730 score points |
| 95% confidence interval of the difference | 1.693 to 1.768 |
| Welch t statistic, degrees of freedom | 89.5, 8,302 |
| p-value (one-sided) | < 1e-300 |
| Effect size, Cohen's d | 1.45 (large; rule of thumb: 0.2 small, 0.5 medium, 0.8 large) |
| Mann-Whitney p-value (check) | < 1e-300 |
| Probability that a random on-time order has a higher score than a random late order (ties count half) | 0.777 |

**Decision: reject H0.** On-time orders score 1.73 points higher on average (4.29 vs 2.56).
The difference is statistically clear and the effect size is large (d = 1.45), so it also matters in practice.
The Mann-Whitney test gives the same answer. *Limit:* this shows a link, not proof that lateness alone causes the low scores.


## 2. One-way ANOVA: does order value differ by payment type? (real data)

- **Question:** Is the average order value (item prices, valid sales) the same for all payment types?
- **H0:** all payment types have the same mean order value.  **H1:** at least one mean is different.
- **Test:** one-way ANOVA. Assumptions: normal values in each group, equal variances, independent orders.
  We check them. Order values are very skewed, so we expect the assumptions to fail and also run
  **Kruskal-Wallis** (does not need normality) as the main result if they do.
- **Data:** valid sales with a payment type (one order has none and is left out).

In [4]:
v = orders[orders.is_valid_sale & orders.main_payment_type.notna()][["main_payment_type", "items_value"]].copy()
v["items_value"] = v.items_value.astype(float)
groups = {k: g.items_value.values for k, g in v.groupby("main_payment_type")}
summary = v.groupby("main_payment_type").items_value.agg(n="count", mean="mean", median="median", sd="std", skew=lambda s: stats.skew(s)).round(2)
display(summary)

# Assumptions
rng = np.random.default_rng(0)
shapiro = {k: stats.shapiro(rng.choice(a, size=min(5000, len(a)), replace=False)).pvalue for k, a in groups.items()}
lev = stats.levene(*groups.values(), center="median")
print("Normality (Shapiro-Wilk on up to 5,000 random orders per group), p-values:", {k: pfmt(p) for k, p in shapiro.items()})
print(f"Equal variances (Levene): statistic = {lev.statistic:.1f}, p = {pfmt(lev.pvalue)}")
normal_ok = all(p >= ALPHA for p in shapiro.values())
var_ok = lev.pvalue >= ALPHA
print("Assumptions OK?", normal_ok and var_ok)

,n,mean,median,sd,skew
main_payment_type,,,,,
boleto,19535,121.71,74.99,201.52,11.69
credit_card,74112,143.67,89.90,213.31,9.31
debit_card,1514,119.62,70.00,226.08,10.75
voucher,3037,94.91,59.99,122.32,5.36


Normality (Shapiro-Wilk on up to 5,000 random orders per group), p-values: {'boleto': '5.9e-84', 'credit_card': '1.53e-79', 'debit_card': '7.03e-59', 'voucher': '6.78e-67'}
Equal variances (Levene): statistic = 53.2, p = 2.33e-34
Assumptions OK? False


In [5]:
anova = stats.f_oneway(*groups.values())
grand = v.items_value.mean()
ss_between = sum(len(a) * (a.mean() - grand) ** 2 for a in groups.values())
ss_total = ((v.items_value - grand) ** 2).sum()
eta2 = ss_between / ss_total

kw = stats.kruskal(*groups.values())
epsilon2 = kw.statistic / (len(v) - 1)

def eta_label(e):
    return "very small" if e < 0.01 else "small" if e < 0.06 else "medium" if e < 0.14 else "large"

tk = stats.tukey_hsd(*groups.values())
names = list(groups)
pairs = []
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        pairs.append({"pair": f"{names[i]} vs {names[j]}", "mean difference": groups[names[i]].mean() - groups[names[j]].mean(),
                      "p (Tukey)": tk.pvalue[i, j]})
pairs = pd.DataFrame(pairs)
pairs["p (Tukey)"] = pairs["p (Tukey)"].map(pfmt)
pairs["mean difference"] = pairs["mean difference"].round(2)

say(f'''
**Result (alpha = {ALPHA})**

| Item | Value |
|---|---|
| ANOVA F statistic (df between, within) | {anova.statistic:.1f} ({len(groups) - 1}, {len(v) - len(groups):,}) |
| ANOVA p-value | {pfmt(anova.pvalue)} |
| Effect size, eta squared | {eta2:.4f} ({eta_label(eta2)}) |
| Kruskal-Wallis H, p-value | {kw.statistic:.1f}, {pfmt(kw.pvalue)} |
| Effect size, epsilon squared (Kruskal-Wallis) | {epsilon2:.4f} ({eta_label(epsilon2)}) |

Pairwise differences (Tukey HSD, raw values):
''')
display(pairs)
say(f'''
**Decision: {decide(anova.pvalue)}** for ANOVA and **{decide(kw.pvalue)}** for Kruskal-Wallis.
The assumptions of ANOVA {'hold' if normal_ok and var_ok else 'do not hold (skewed values, unequal variances)'}, so the Kruskal-Wallis result is the one to trust;
both tests agree. Payment types differ in order value, but the effect is {eta_label(max(eta2, epsilon2))}: payment type explains about
{100 * eta2:.1f}% of the variation in order value (eta squared). For example, voucher orders average {summary.loc['voucher', 'mean']:.2f}
and credit card orders {summary.loc['credit_card', 'mean']:.2f}. *Limit:* the Tukey post-test also assumes normal data, so read it as a guide only.
''')


**Result (alpha = 0.05)**

| Item | Value |
|---|---|
| ANOVA F statistic (df between, within) | 104.4 (3, 98,194) |
| ANOVA p-value | 1.71e-67 |
| Effect size, eta squared | 0.0032 (very small) |
| Kruskal-Wallis H, p-value | 948.0, 3.44e-205 |
| Effect size, epsilon squared (Kruskal-Wallis) | 0.0097 (very small) |

Pairwise differences (Tukey HSD, raw values):


,pair,mean difference,p (Tukey)
0,boleto vs credit_card,-21.96,< 1e-300
1,boleto vs debit_card,2.08,0.982
2,boleto vs voucher,26.80,2.73e-10
3,credit_card vs debit_card,24.04,5.54e-05
4,credit_card vs voucher,48.76,< 1e-300
5,debit_card vs voucher,24.71,0.000981



**Decision: reject H0** for ANOVA and **reject H0** for Kruskal-Wallis.
The assumptions of ANOVA do not hold (skewed values, unequal variances), so the Kruskal-Wallis result is the one to trust;
both tests agree. Payment types differ in order value, but the effect is very small: payment type explains about
0.3% of the variation in order value (eta squared). For example, voucher orders average 94.91
and credit card orders 143.67. *Limit:* the Tukey post-test also assumes normal data, so read it as a guide only.


## 3. Chi-square test: does the RFM segment depend on the region? (real data)

- **Question:** Is the share of each RFM segment (Champions, Loyal, ...) the same in all 5 Brazilian regions?
- **H0:** segment and region are independent.  **H1:** they are not independent.
- **Test:** chi-square test of independence on the segment x region table (Q12 plus the customer state).
  Assumptions: each customer appears once (yes: one row per customer_unique_id) and expected counts are at least 5 (checked below).
- **Effect size:** Cramer's V (0 = no link, 1 = perfect link).

In [6]:
REGION = {}
for region, states in {"North": "AC AP AM PA RO RR TO", "Northeast": "AL BA CE MA PB PE PI RN SE",
                       "Central-West": "DF GO MT MS", "Southeast": "ES MG RJ SP", "South": "PR RS SC"}.items():
    for s in states.split():
        REGION[s] = region

rfm = q("q12")
rfm["region"] = rfm.customer_state.map(REGION)
assert rfm.region.notna().all()
table = pd.crosstab(rfm.segment, rfm.region)
display(table)
print("Share of each segment inside each region (%):")
display((table / table.sum() * 100).round(1))

chi2, p, dof, expected = stats.chi2_contingency(table)
n = table.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(table.shape) - 1)))
print("smallest expected count:", round(expected.min(), 1), "(must be at least 5)")

def v_label(v):
    return "very small" if v < 0.1 else "small" if v < 0.3 else "medium" if v < 0.5 else "large"

say(f'''
**Result (alpha = {ALPHA})**

| Item | Value |
|---|---|
| Chi-square statistic (degrees of freedom) | {chi2:.1f} ({dof}) |
| p-value | {pfmt(p)} |
| Effect size, Cramer's V | {cramers_v:.3f} ({v_label(cramers_v)}) |

**Decision: {decide(p)}.** The segment mix is not exactly the same in all regions, but Cramer's V is only {cramers_v:.3f}
({v_label(cramers_v)}): region explains very little about the segment a customer belongs to.
With {n:,} customers even a tiny difference is "significant", so the business conclusion is that region is **not** a useful way to
predict customer value. *Limit:* RFM scores are partly built from recency, so segments depend on when the data was cut.
''')

region,Central-West,North,Northeast,South,Southeast
segment,,,,,
About to Sleep,1074,370,1894,2661,12423
At Risk,64,20,91,148,707
Champions,48,13,68,137,692
Lost,1032,434,1898,2871,12307
Loyal,55,16,60,114,654
Needs Attention,1173,338,1745,2664,12459
New Customers,2097,588,3297,4956,25815


Share of each segment inside each region (%):


region,Central-West,North,Northeast,South,Southeast
segment,,,,,
About to Sleep,19.4,20.8,20.9,19.6,19.1
At Risk,1.2,1.1,1.0,1.1,1.1
Champions,0.9,0.7,0.8,1.0,1.1
Lost,18.6,24.4,21.0,21.2,18.9
Loyal,1.0,0.9,0.7,0.8,1.0
Needs Attention,21.2,19.0,19.3,19.7,19.2
New Customers,37.8,33.1,36.4,36.6,39.7


smallest expected count: 16.8 (must be at least 5)



**Result (alpha = 0.05)**

| Item | Value |
|---|---|
| Chi-square statistic (degrees of freedom) | 175.6 (24) |
| p-value | 4.95e-25 |
| Effect size, Cramer's V | 0.022 (very small) |

**Decision: reject H0.** The segment mix is not exactly the same in all regions, but Cramer's V is only 0.022
(very small): region explains very little about the segment a customer belongs to.
With 94,983 customers even a tiny difference is "significant", so the business conclusion is that region is **not** a useful way to
predict customer value. *Limit:* RFM scores are partly built from recency, so segments depend on when the data was cut.


## 4. Probability from real data

- **P(customer buys again)** = customers with 2 or more valid orders / all customers with a valid order (customer_unique_id).
- **P(low review | late)** and **P(low review | on time)**: low review = score 1 or 2. Delivered orders with a review.
- Two-proportion z-test for the two conditional probabilities. H0: they are equal. H1: P(low | late) is higher.

In [7]:
from statsmodels.stats.proportion import confint_proportions_2indep, proportion_confint, proportions_ztest

valid = orders[orders.is_valid_sale]
n_orders = valid.groupby("customer_unique_id").size()
buy_again = int((n_orders >= 2).sum())
p_again = buy_again / len(n_orders)
lo, hi = proportion_confint(buy_again, len(n_orders), method="wilson")
q10 = q("q10").set_index("metric").value
assert buy_again == int(q10["repeat_customers"])   # matches Q10

d["low"] = d.review_score <= 2
d["late"] = d.delay_days > 0
low_late, n_late = int(d[d.late].low.sum()), int(d.late.sum())
low_on, n_on = int(d[~d.late].low.sum()), int((~d.late).sum())
p_low_late, p_low_on = low_late / n_late, low_on / n_on
rr = p_low_late / p_low_on
se_log_rr = np.sqrt(1 / low_late - 1 / n_late + 1 / low_on - 1 / n_on)
rr_lo, rr_hi = np.exp(np.log(rr) + np.array([-1, 1]) * 1.96 * se_log_rr)
dci = confint_proportions_2indep(low_late, n_late, low_on, n_on, method="wald")
z, pz = proportions_ztest([low_late, low_on], [n_late, n_on], alternative="larger")

# Bayes check: P(late | low) from P(low | late) * P(late) / P(low)
p_late = n_late / len(d)
p_low = d.low.mean()
p_late_given_low = p_low_late * p_late / p_low
direct = d[d.low].late.mean()
assert abs(p_late_given_low - direct) < 1e-9

say(f'''
| Probability | Value |
|---|---|
| P(customer buys again) | {p_again:.4f} ({100 * p_again:.2f}%), 95% CI {100 * lo:.2f}% to {100 * hi:.2f}% ({buy_again:,} of {len(n_orders):,}; same as Q10) |
| P(order is late) | {p_late:.4f} ({100 * p_late:.2f}%) |
| P(low review given late) | {p_low_late:.4f} ({100 * p_low_late:.2f}%) |
| P(low review given on time) | {p_low_on:.4f} ({100 * p_low_on:.2f}%) |
| Difference (late minus on time), 95% CI | {100 * (p_low_late - p_low_on):.2f} points ({100 * dci[0]:.2f} to {100 * dci[1]:.2f}) |
| Risk ratio, 95% CI | {rr:.2f} ({rr_lo:.2f} to {rr_hi:.2f}) |
| Two-proportion z-test p-value (one-sided) | {pfmt(pz)} |
| P(late given low review), by Bayes' rule (equals the direct count) | {p_late_given_low:.4f} |

**Decision: {decide(pz)}.** A late order is {rr:.1f} times as likely to get a low review as an on-time order.
Only about {100 * p_again:.1f}% of customers buy again, so each first order is worth a lot, and a late delivery makes a bad first impression.
Also, {100 * p_late_given_low:.0f}% of all low reviews come from late orders, although late orders are only {100 * p_late:.0f}% of orders.
*Limit:* a link, not proof of cause.
''')


| Probability | Value |
|---|---|
| P(customer buys again) | 0.0304 (3.04%), 95% CI 2.93% to 3.15% (2,887 of 94,983; same as Q10) |
| P(order is late) | 0.0799 (7.99%) |
| P(low review given late) | 0.5411 (54.11%) |
| P(low review given on time) | 0.0922 (9.22%) |
| Difference (late minus on time), 95% CI | 44.89 points (43.75 to 46.02) |
| Risk ratio, 95% CI | 5.87 (5.70 to 6.04) |
| Two-proportion z-test p-value (one-sided) | < 1e-300 |
| P(late given low review), by Bayes' rule (equals the direct count) | 0.3375 |

**Decision: reject H0.** A late order is 5.9 times as likely to get a low review as an on-time order.
Only about 3.0% of customers buy again, so each first order is worth a lot, and a late delivery makes a bad first impression.
Also, 34% of all low reviews come from late orders, although late orders are only 8% of orders.
*Limit:* a link, not proof of cause.


## 5. A/B test: free shipping vs paid shipping (**SIMULATED DATA**)

> **SIMULATED.** Olist has no A/B test data. Everything in this section is **made up by a computer with a fixed random seed**.
> The conversion rates, the visitors and the results are **not real customers**. The goal is to show the method:
> power analysis, sample size, z-test, confidence interval, lift and a recommendation.

- **Scenario:** at checkout, group A pays for shipping (control), group B gets free shipping (treatment).
- **Metric:** checkout conversion = share of checkout visitors who complete the purchase.
- **H0:** conversion rate of B = conversion rate of A.  **H1:** they differ (two-sided).
- **Test:** two-proportion z-test (two groups, yes/no outcome, large samples). alpha = 0.05.

In [8]:
# ---- SIMULATED assumptions (stated, not measured) ----
SEED = 42                 # fixed random seed, so the result is the same on every run
SIM_BASE_RATE = 0.030     # SIMULATED assumption: 3.0% of checkout visitors buy with paid shipping (group A)
SIM_MDE = 0.005           # smallest effect worth detecting: +0.5 percentage points (3.0% -> 3.5%)
SIM_TRUE_RATE_B = SIM_BASE_RATE + SIM_MDE   # SIMULATED assumption: the real effect of free shipping equals the MDE
POWER = 0.80

from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

effect = proportion_effectsize(SIM_TRUE_RATE_B, SIM_BASE_RATE)   # Cohen's h
n_per_group = int(np.ceil(NormalIndPower().solve_power(effect, alpha=ALPHA, power=POWER, ratio=1, alternative="two-sided")))

say(f'''
**SIMULATED power analysis**

| Item | Value |
|---|---|
| Base conversion rate of A (assumption) | {100 * SIM_BASE_RATE:.1f}% |
| Minimum detectable effect (MDE) | +{100 * SIM_MDE:.1f} percentage points ({100 * SIM_MDE / SIM_BASE_RATE:.1f}% relative lift) |
| Effect size (Cohen's h) | {effect:.4f} |
| alpha, power | {ALPHA}, {POWER:.0%} |
| **Sample size needed per group** | **{n_per_group:,}** checkout visitors ({2 * n_per_group:,} in total) |
''')


**SIMULATED power analysis**

| Item | Value |
|---|---|
| Base conversion rate of A (assumption) | 3.0% |
| Minimum detectable effect (MDE) | +0.5 percentage points (16.7% relative lift) |
| Effect size (Cohen's h) | 0.0282 |
| alpha, power | 0.05, 80% |
| **Sample size needed per group** | **19,716** checkout visitors (39,432 in total) |


In [9]:
from statsmodels.stats.proportion import confint_proportions_2indep, proportions_ztest

rng = np.random.default_rng(SEED)
conv_a = int(rng.binomial(n_per_group, SIM_BASE_RATE))       # SIMULATED conversions in group A
conv_b = int(rng.binomial(n_per_group, SIM_TRUE_RATE_B))     # SIMULATED conversions in group B
rate_a, rate_b = conv_a / n_per_group, conv_b / n_per_group

z, p = proportions_ztest([conv_b, conv_a], [n_per_group, n_per_group], alternative="two-sided")
ci_lo, ci_hi = confint_proportions_2indep(conv_b, n_per_group, conv_a, n_per_group, method="wald")
lift = 100 * (rate_b - rate_a) / rate_a

say(f'''
**SIMULATED result (seed {SEED})**

| Item | A: paid shipping | B: free shipping |
|---|---|---|
| Visitors | {n_per_group:,} | {n_per_group:,} |
| Conversions | {conv_a:,} | {conv_b:,} |
| Conversion rate | {100 * rate_a:.2f}% | {100 * rate_b:.2f}% |

| Item | Value |
|---|---|
| Difference B minus A | {100 * (rate_b - rate_a):.2f} percentage points |
| 95% confidence interval of the difference | {100 * ci_lo:.2f} to {100 * ci_hi:.2f} percentage points |
| Lift (relative) | {lift:.1f}% |
| z statistic, p-value (two-sided) | {z:.2f}, {pfmt(p)} |

**Decision: {decide(p)}** (SIMULATED data).

The observed lift ({lift:.1f}%) is {'larger' if lift > 100 * SIM_MDE / SIM_BASE_RATE else 'smaller'} than the lift built into the simulation ({100 * SIM_MDE / SIM_BASE_RATE:.1f}%). This is random noise, and it shows why one experiment can over- or under-state an effect: look at the confidence interval, not only at the single number.
''')


**SIMULATED result (seed 42)**

| Item | A: paid shipping | B: free shipping |
|---|---|---|
| Visitors | 19,716 | 19,716 |
| Conversions | 571 | 704 |
| Conversion rate | 2.90% | 3.57% |

| Item | Value |
|---|---|
| Difference B minus A | 0.67 percentage points |
| 95% confidence interval of the difference | 0.33 to 1.02 percentage points |
| Lift (relative) | 23.3% |
| z statistic, p-value (two-sided) | 3.79, 0.000153 |

**Decision: reject H0** (SIMULATED data).

The observed lift (23.3%) is larger than the lift built into the simulation (16.7%). This is random noise, and it shows why one experiment can over- or under-state an effect: look at the confidence interval, not only at the single number.


In [10]:
# How reliable is the design? Repeat the SIMULATED experiment 5,000 times (another fixed seed).
rng2 = np.random.default_rng(123)
runs = 5000

def reject_rate(true_a, true_b):
    a = rng2.binomial(n_per_group, true_a, runs) / n_per_group
    b = rng2.binomial(n_per_group, true_b, runs) / n_per_group
    pooled = (a + b) / 2
    zs = (b - a) / np.sqrt(pooled * (1 - pooled) * 2 / n_per_group)
    return float(np.mean(np.abs(zs) > stats.norm.ppf(1 - ALPHA / 2)))

power_sim = reject_rate(SIM_BASE_RATE, SIM_TRUE_RATE_B)
false_pos = reject_rate(SIM_BASE_RATE, SIM_BASE_RATE)
say(f'''
**SIMULATED check of the design ({runs:,} repeated experiments):**
if free shipping really adds +{100 * SIM_MDE:.1f} points, the test finds it in **{100 * power_sim:.1f}%** of experiments (target {POWER:.0%}).
If free shipping has no effect, the test wrongly finds one in **{100 * false_pos:.1f}%** of experiments (target {100 * ALPHA:.0f}%).
This is why a single experiment can still miss a real effect: power is 80%, not 100%.
''')


**SIMULATED check of the design (5,000 repeated experiments):**
if free shipping really adds +0.5 points, the test finds it in **80.7%** of experiments (target 80%).
If free shipping has no effect, the test wrongly finds one in **5.1%** of experiments (target 5%).
This is why a single experiment can still miss a real effect: power is 80%, not 100%.


In [11]:
# Real data for the cost side: what does shipping cost per order today? (REAL, from the Olist orders)
sales = orders[orders.is_valid_sale]
avg_freight, avg_items = float(sales.freight_value.astype(float).mean()), float(sales.items_value.astype(float).mean())
say(f'''
**Real cost context (Olist data, not simulated):** valid orders have an average item value of {avg_items:.2f} BRL and an average freight
of {avg_freight:.2f} BRL, so freight is {100 * avg_freight / avg_items:.1f}% of the item value. Free shipping means the company pays this
amount (or part of it), so it must win enough extra orders to cover it.

### Recommendation (based on SIMULATED results)
{'The simulated test shows a clear lift in conversion, and the 95% confidence interval stays above zero.' if ci_lo > 0 else 'The simulated test does not show a clear lift: the 95% confidence interval includes zero.'}
Do **not** decide on conversion alone: a +{100 * (rate_b - rate_a):.2f} point conversion gain only pays off if the extra margin from the
extra orders is larger than the shipping cost of **all** orders in group B (including the ones that would have bought anyway).

### What a real test would also need
- **Duration:** run for at least 2 full weeks (whole weekly cycles; Q5 shows weekdays and weekends behave differently), and stop only when
  the planned sample size is reached. Do not stop early when the p-value looks good.
- **Random assignment** by visitor, and a check that the two groups have the planned sizes (sample ratio mismatch).
- **Novelty effect:** free shipping may excite visitors at first. Compare week 1 with later weeks.
- **Guardrail metrics:** margin per order, average order value, shipping cost per order, delivery time and review score, returns/cancellations.
- **Decision rule written before the test:** the minimum margin gain that makes free shipping worth it.
''')


**Real cost context (Olist data, not simulated):** valid orders have an average item value of 137.42 BRL and an average freight
of 22.82 BRL, so freight is 16.6% of the item value. Free shipping means the company pays this
amount (or part of it), so it must win enough extra orders to cover it.

### Recommendation (based on SIMULATED results)
The simulated test shows a clear lift in conversion, and the 95% confidence interval stays above zero.
Do **not** decide on conversion alone: a +0.67 point conversion gain only pays off if the extra margin from the
extra orders is larger than the shipping cost of **all** orders in group B (including the ones that would have bought anyway).

### What a real test would also need
- **Duration:** run for at least 2 full weeks (whole weekly cycles; Q5 shows weekdays and weekends behave differently), and stop only when
  the planned sample size is reached. Do not stop early when the p-value looks good.
- **Random assignment** by visitor, and a check that the two groups have the planned sizes (sample ratio mismatch).
- **Novelty effect:** free shipping may excite visitors at first. Compare week 1 with later weeks.
- **Guardrail metrics:** margin per order, average order value, shipping cost per order, delivery time and review score, returns/cancellations.
- **Decision rule written before the test:** the minimum margin gain that makes free shipping worth it.


## Summary

| Test | Data | Result in one line |
|---|---|---|
| Welch t-test, review score late vs on time | real | see Part 1: on-time orders score clearly higher, large effect |
| ANOVA / Kruskal-Wallis, order value by payment type | real | see Part 2: payment types differ, very small effect |
| Chi-square, RFM segment vs region | real | see Part 3: tiny effect, region does not predict segment |
| Probabilities | real | see Part 4: few customers buy again, late orders get many more low reviews |
| A/B test, free shipping | **SIMULATED** | see Part 5: method demo only, not real customers |

## Save the key results for the report

In [12]:
import json

results = {
    "source": "notebooks/03_statistics_ab_test.ipynb",
    "welch_ttest_real": {
        "n_on_time": int(len(ontime)), "n_late": int(len(late)), "mean_on_time": float(ontime.mean()), "mean_late": float(late.mean()),
        "difference": float(diff), "ci95_low": float(ci.low), "ci95_high": float(ci.high), "cohen_d": float(cohen_d),
        "p_value": float(res.pvalue), "alpha": ALPHA, "decision": decide(res.pvalue),
    },
    "anova_real": {"anova_p": float(anova.pvalue), "eta_squared": float(eta2), "kruskal_p": float(kw.pvalue),
                   "epsilon_squared": float(epsilon2), "assumptions_hold": bool(normal_ok and var_ok),
                   "mean_voucher": float(summary.loc["voucher", "mean"]), "mean_credit_card": float(summary.loc["credit_card", "mean"])},
    "chi_square_real": {"chi2": float(chi2), "p_value": float(p), "cramers_v": float(cramers_v), "n_customers": int(n)},
    "probability_real": {"p_buy_again": float(p_again), "p_buy_again_ci_low": float(lo), "p_buy_again_ci_high": float(hi),
                         "p_late": float(p_late), "p_low_given_late": float(p_low_late), "p_low_given_on_time": float(p_low_on),
                         "risk_ratio": float(rr), "risk_ratio_ci_low": float(rr_lo), "risk_ratio_ci_high": float(rr_hi),
                         "p_late_given_low_review": float(p_late_given_low)},
    "ab_test_SIMULATED": {"label": "SIMULATED DATA, not real customers", "seed": SEED, "assumed_base_rate": SIM_BASE_RATE,
                          "assumed_mde": SIM_MDE, "alpha": ALPHA, "power": POWER, "sample_size_per_group": int(n_per_group),
                          "conversions_a": conv_a, "conversions_b": conv_b, "rate_a": float(rate_a), "rate_b": float(rate_b),
                          "lift_pct": float(lift), "p_value": float(p), "ci95_low_points": float(100 * ci_lo),
                          "ci95_high_points": float(100 * ci_hi), "simulated_power": float(power_sim),
                          "simulated_false_positive_rate": float(false_pos)},
    "real_freight_context": {"avg_items_value": float(avg_items), "avg_freight": float(avg_freight),
                             "freight_pct_of_items_value": float(100 * avg_freight / avg_items)},
}
(ROOT / "reports" / "stats_results.json").write_text(json.dumps(results, indent=2), encoding="utf-8")
print("saved reports/stats_results.json")

saved reports/stats_results.json
